In [2]:
# Cell 1: Upload kaggle.json and download dataset
from google.colab import files
import os, zipfile

print("Please upload your kaggle.json file:")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download New Plant Diseases Dataset (Pre-split train & valid)
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset

print("Extracting dataset...")
with zipfile.ZipFile("new-plant-diseases-dataset.zip", "r") as zip_ref:
    zip_ref.extractall("plant_data")

print("Done! Checking directories...")
base_dir = "plant_data/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)"
print("Train classes:", len(os.listdir(os.path.join(base_dir, "train"))))
print("Valid classes:", len(os.listdir(os.path.join(base_dir, "valid"))))

Please upload your kaggle.json file:


KeyboardInterrupt: 

In [3]:
# Cell 2: Train MobileNetV2 & Export to ONNX + labels.json
!pip install -q onnx onnxruntime

import os, json, time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

base_dir = "plant_data/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)"
train_dir = os.path.join(base_dir, "train")
valid_dir = os.path.join(base_dir, "valid")

# ImageNet standard normalization (CRITICAL: Must match Next.js preprocessing!)
data_transforms = {
    "train": transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    "valid": transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

train_dataset = datasets.ImageFolder(train_dir, transform=data_transforms["train"])
valid_dataset = datasets.ImageFolder(valid_dir, transform=data_transforms["valid"])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False, num_workers=2)

class_names = train_dataset.classes
num_classes = len(class_names)
print(f"Total Classes: {num_classes} | Train Images: {len(train_dataset)} | Valid Images: {len(valid_dataset)}")

# Save class index mapping for Next.js
idx_to_class = {str(i): name for i, name in enumerate(class_names)}
with open("labels.json", "w") as f:
    json.dump(idx_to_class, f, indent=2)

# Load Pretrained MobileNetV2
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.last_channel, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

# Train for 3 Epochs (Reaches ~95%+ accuracy on PlantVillage)
epochs = 3
for epoch in range(epochs):
    start_time = time.time()
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = 100. * correct / total

    # Validation
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_acc = 100. * val_correct / val_total
    print(f"Epoch [{epoch+1}/{epochs}] - Time: {time.time()-start_time:.1f}s - Train Acc: {train_acc:.2f}% - Val Acc: {val_acc:.2f}%")

# Export to ONNX (Opset 14 for maximum Node.js compatibility)
model.eval()
dummy_input = torch.randn(1, 3, 224, 224, device=device)
onnx_path = "crop_disease_model.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}
)
print(f"Model successfully exported to {onnx_path}!")

Using device: cuda
Total Classes: 38 | Train Images: 70295 | Valid Images: 17572
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 104MB/s] 


Epoch [1/3] - Time: 366.8s - Train Acc: 95.44% - Val Acc: 99.39%
Epoch [2/3] - Time: 360.0s - Train Acc: 99.01% - Val Acc: 99.18%
Epoch [3/3] - Time: 360.5s - Train Acc: 99.35% - Val Acc: 99.64%


ModuleNotFoundError: No module named 'onnxscript'

In [4]:
!pip install -q onnxscript

model.eval()
dummy_input = torch.randn(1, 3, 224, 224, device=device)
onnx_path = "crop_disease_model.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}
)
print(f"Model successfully exported to {onnx_path}!")

/tmp/ipykernel_2343/3661824524.py:7: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W1114 13:49:26.298000 2343 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
RuntimeError: /project/onnx/version_converter/adapters/axes_input_to_attribute.h:55: 

[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Model successfully exported to crop_disease_model.onnx!


In [5]:
# Cell 3: Verify ONNX output & Download files
import onnxruntime as ort
import numpy as np
from google.colab import files

session = ort.InferenceSession("crop_disease_model.onnx")
test_input = np.random.randn(1, 3, 224, 224).astype(np.float32)
outputs = session.run(None, {"input": test_input})

print("ONNX Verification Passed! Output shape:", outputs[0].shape)

# Download all necessary files to your PC
files.download("crop_disease_model.onnx")
files.download("crop_disease_model.onnx.data")
files.download("labels.json")

ONNX Verification Passed! Output shape: (1, 38)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>